In [8]:
import torch
from torch import nn
from torchvision.transforms import ToPILImage
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

def tensor_to_PILimage(net: nn.Module, 
                       train_iter: DataLoader | torch.Tensor,
                       tensor_index: int=0,
                       columns: int=8,
                       figsize: tuple=(2,2),
                      ) -> None:
    '''
    tensor_index: the index of the tensor to choose
    columns: the number of plots in one row
    figsize: tuple of (width,height)
    '''
    if isinstance(net,nn.DataParallel):
        net=net.module.cpu()
        
    if isinstance(train_iter,DataLoader):
        X=train_iter.dataset[tensor_index][0]
        #y=train_iter.dataset[tensor_index][1]
    elif isinstance(train_iter,torch.Tensor):
        X=train_iter
    else:
        raise "Wrong train_iter"
        
    plot_width,plot_height=figsize
    fig, axes = plt.subplots(1, 1, figsize=(2, 2))
    image_X=ToPILImage()(X)
    axes.imshow(image_X)
    
    layer=0
    y_hat=net[layer](X)
    to_pil=ToPILImage()
    while not isinstance(net[layer],nn.Flatten):
        chanel=y_hat.shape[0]
        #height=y_hat.shape[1]
        #width=y_hat.shape[2]
        row=int((chanel-0.9)//columns+1)  #否则恰好整除时会多 1 
        fig, axes = plt.subplots(row, columns,
                                 figsize=(plot_width*columns, plot_height*row))
        for i in range(chanel):
            image = to_pil(y_hat[[i]])
            if row==1:
                axes[i].imshow(image)
                axes[0].set(title=f'net{layer} '+type(net[layer]).__name__)
            else:
                axes[i//columns,i%columns].imshow(image) #实际上是灰度的,是matplotlib带了颜色
                axes[0,0].set(title=f'net{layer} '+type(net[layer]).__name__)
        layer+=1
        y_hat=net[layer](y_hat)
